# Hindcast 0008-01: Z300 Maps And Wave Amplitudes

拆出 Z300 pointwise correlation maps 和 Z300 zonal-wavenumber amplitude source-hunting diagnostics。

In [ ]:
from __future__ import annotations

import glob
import math
import os
import re
import warnings
from pathlib import Path
from typing import Dict, Iterable, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
from scipy.stats import linregress, pearsonr

plt.rcParams.update({
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.grid": True,
    "grid.linestyle": "--",
    "grid.alpha": 0.35,
})

CASE = "0008-01"
REF_YEAR = 8

REPO_ROOT = Path("/home/weiji/restart_exam/code_cleaned")
TEST_ROOT = REPO_ROOT / "Hindcast_experiment" / "TEST_TROPOS"
OUT_ROOT = TEST_ROOT / "outputs" / CASE
FIG_DIR = OUT_ROOT / "figures"
TABLE_DIR = OUT_ROOT / "tables"
CACHE_DIR = OUT_ROOT / "cache"
for d in [FIG_DIR, TABLE_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DATA_ROOT = Path("/mnt/soclim0/public_data/weiji")
HINDCAST_ROOT = DATA_ROOT / "Hindcast"
BWCN_ROOT = DATA_ROOT / "BWCN"
B2000_ROOT = DATA_ROOT / "B2000WCN001002_timefixed"

CASE_ROOT = HINDCAST_ROOT / CASE

# Main windows. The EP window matches the old Hindcast_vertical_analysis scatter.
O3_END = (5, 30)
EP_WINDOW = ((1, 20), (2, 10))
EP_WINDOW_LABEL = "Jan20-Feb10"
AO_BASE_WINDOW = EP_WINDOW
AO_BASE_WINDOW_LABEL = EP_WINDOW_LABEL
AO_TEST_WINDOWS = {
    "Jan20-Feb10": EP_WINDOW,
    "Jan": ((1, 1), (1, 31)),
    "Jan-Feb": ((1, 1), (2, 28)),
    "Jan-Mar": ((1, 1), (3, 31)),
}
Z300_WINDOW = EP_WINDOW
Z300_WINDOW_LABEL = EP_WINDOW_LABEL
MONTH_WINDOWS = {
    "Jan": ((1, 1), (1, 31)),
    "Feb": ((2, 1), (2, 28)),
    "Mar": ((3, 1), (3, 31)),
    "Apr": ((4, 1), (4, 30)),
    "May": ((5, 1), (5, 30)),
}
MONTH_ORDER = ["Jan", "Feb", "Mar", "Apr", "May"]

LAT_EP = (40.0, 80.0)
LAT_POLAR = (60.0, 90.0)
LAT_Z300 = (20.0, 90.0)
PLEV_EP_PA = 10000.0
PLEV_EP_HPA = PLEV_EP_PA / 100.0
PLEV_U_PA = 1000.0
PLEV_Z300_PA = 30000.0
PLEV_Z300_HPA = PLEV_Z300_PA / 100.0

EP_SCALAR_DESCRIPTION = (
    f"mean -ep2 vertical EP-flux component at {PLEV_EP_HPA:.0f} hPa, "
    f"cos-lat mean {LAT_EP[0]:.0f}-{LAT_EP[1]:.0f}N, {EP_WINDOW_LABEL}; not EP-flux divergence"
)
Z300_PATTERN_DESCRIPTION = (
    f"{PLEV_Z300_HPA:.0f} hPa height, {Z300_WINDOW_LABEL} mean, "
    f"{LAT_Z300[0]:.0f}-{LAT_Z300[1]:.0f}N, zonal mean removed before pattern metrics"
)

WAVES = ["all_waves", "wave1", "wave2", "wave_rest"]
WAVE_LABELS = {
    "all_waves": "All waves",
    "wave1": "Wave 1",
    "wave2": "Wave 2",
    "wave_rest": "Wave rest",
    "wave1_plus_wave2": "Wave 1 + Wave 2",
}

# Expensive sections cache their results. Keep True for the full requested test.
RUN_Z300_DIAGNOSTICS = True
BUILD_U60N10_IF_MISSING = True
CLIM_MAX_B2000_YEARS_FOR_Z300 = None  # None = all B2000WCN001002 Z3 years available.

print(f"Output root: {OUT_ROOT}")
print(f"Case root exists: {CASE_ROOT.exists()} -> {CASE_ROOT}")

In [ ]:
# -----------------------------
# Shared helpers
# -----------------------------

def member_short_id(member) -> str:
    text = str(member)
    m = re.search(r"\.(\d{3})\.cam\.h3", text)
    if m:
        return m.group(1)
    m = re.search(r"\.(\d{3})\.", text)
    return m.group(1) if m else text


def date_parts(date_values):
    arr = np.asarray(date_values, dtype=np.int64)
    year = arr // 10000
    mmdd = arr % 10000
    month = mmdd // 100
    day = mmdd % 100
    return year, month, day


def date_mask(date_values, start=(1, 1), end=(5, 30), year: Optional[int] = None):
    yy, mm, dd = date_parts(date_values)
    start_key = start[0] * 100 + start[1]
    end_key = end[0] * 100 + end[1]
    key = mm * 100 + dd
    mask = (key >= start_key) & (key <= end_key)
    if year is not None:
        mask = mask & (yy == int(year))
    return mask


def one_dim_date(ds_or_da) -> np.ndarray:
    date = ds_or_da["date"]
    if "member" in date.dims:
        date = date.isel(member=0)
    return np.asarray(date.values, dtype=np.int64)


def assign_member_short_coord(da: xr.DataArray) -> xr.DataArray:
    if "member" not in da.dims:
        return da
    mids = [member_short_id(v) for v in da["member"].values]
    return da.assign_coords(member_short=("member", mids))


def select_latband(da: xr.DataArray, lat_range: Tuple[float, float], lat_name="lat") -> xr.DataArray:
    lat = da[lat_name]
    descending = float(lat.values[0]) > float(lat.values[-1])
    lo, hi = lat_range
    return da.sel({lat_name: slice(hi, lo) if descending else slice(lo, hi)})


def coslat_mean(da: xr.DataArray, lat_range: Optional[Tuple[float, float]] = None, lat_name="lat") -> xr.DataArray:
    if lat_range is not None:
        da = select_latband(da, lat_range, lat_name=lat_name)
    weights = np.cos(np.deg2rad(da[lat_name])).clip(0, 1)
    return da.weighted(weights.fillna(0)).mean(lat_name, skipna=True)


def finite_corr(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3:
        return np.nan, np.nan
    r, p = pearsonr(x[mask], y[mask])
    return float(r), float(p)


def scatter_fit(ax, df, xcol, ycol, title, xlabel=None, ylabel=None, color="tab:blue", annotate_members=True):
    sub = df[[xcol, ycol, "member"]].replace([np.inf, -np.inf], np.nan).dropna()
    ax.scatter(sub[xcol], sub[ycol], s=62, color=color, edgecolor="black", linewidth=0.5, alpha=0.88)
    if annotate_members:
        for _, row in sub.iterrows():
            ax.text(row[xcol], row[ycol], str(row["member"]), fontsize=7, alpha=0.65)
    if len(sub) >= 3:
        fit = linregress(sub[xcol].values, sub[ycol].values)
        xx = np.linspace(sub[xcol].min(), sub[xcol].max(), 100)
        ax.plot(xx, fit.slope * xx + fit.intercept, color="crimson", ls="--", lw=1.8)
        ax.text(
            0.03, 0.97,
            f"R = {fit.rvalue:.3f}\nP = {fit.pvalue:.2e}",
            transform=ax.transAxes,
            va="top",
            ha="left",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.84, edgecolor="0.7"),
        )
    ax.set_title(title)
    ax.set_xlabel(xlabel or xcol)
    ax.set_ylabel(ylabel or ycol)
    return ax


def savefig(fig, name):
    png = FIG_DIR / f"{name}.png"
    pdf = FIG_DIR / f"{name}.pdf"
    fig.savefig(png, dpi=300, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    print(f"Saved: {png}")
    return png, pdf


def mmdd_label(date_values):
    _, mm, dd = date_parts(date_values)
    return np.array([f"{int(m):02d}-{int(d):02d}" for m, d in zip(mm, dd)])


def month_ticks(date_values):
    _, mm, dd = date_parts(date_values)
    positions, labels = [], []
    names = {1:"Jan", 2:"Feb", 3:"Mar", 4:"Apr", 5:"May", 6:"Jun"}
    seen = set()
    for i, (m, d) in enumerate(zip(mm, dd)):
        if int(d) == 1 and int(m) not in seen:
            positions.append(i)
            labels.append(names.get(int(m), str(int(m))))
            seen.add(int(m))
    return positions, labels


def standardize_1d(y):
    y = np.asarray(y, dtype=float)
    return (y - np.nanmean(y)) / np.nanstd(y)

In [ ]:
# -----------------------------
# Data loaders for the cleaned Hindcast products
# -----------------------------

def load_hindcast_o3(case=CASE, pressure_tag="30_70hPa"):
    path = HINDCAST_ROOT / case / "partial_O3" / f"{case}_partial_O3_all_ranges_members.nc"
    if not path.exists():
        raise FileNotFoundError(f"Missing cleaned partial O3 product: {path}")
    with xr.open_dataset(path, decode_times=False) as ds:
        var = f"O3_partial_60_90N_{pressure_tag}"
        da = assign_member_short_coord(ds[var]).load()
        date = one_dim_date(ds)
    mask = date_mask(date, start=(1, 1), end=O3_END)
    da = da.isel(lead_time=mask)
    date = date[mask]
    da = da.assign_coords(date=("lead_time", date))
    return da, date


def load_bwcn_ref_o3(year=REF_YEAR, pressure_tag="30_70hPa"):
    path = BWCN_ROOT / "partial_O3" / "BWCN_partial_O3_all_ranges.nc"
    if not path.exists():
        raise FileNotFoundError(path)
    with xr.open_dataset(path, decode_times=False) as ds:
        var = f"O3_partial_60_90N_{pressure_tag}"
        date = np.asarray(ds["date"].values, dtype=np.int64)
        mask = date_mask(date, start=(1, 1), end=O3_END, year=year)
        da = ds[var].isel(time=mask).load()
        date = date[mask]
    da = da.rename({"time": "lead_time"}).assign_coords(lead_time=np.arange(len(date)), date=("lead_time", date))
    return da, date


def load_epflux_wave(case=CASE, wave="all_waves", plev_pa=PLEV_EP_PA, lat_range=LAT_EP):
    path = HINDCAST_ROOT / case / "EPflux_daily_ubar" / wave / f"EPFLUX_{wave}_{case}_members_time_plev_lat.nc"
    if not path.exists():
        raise FileNotFoundError(path)
    with xr.open_dataset(path, decode_times=False) as ds:
        ep2 = assign_member_short_coord(ds["ep2"])
        ep2_100 = ep2.sel(plev=plev_pa, method="nearest")
        ep2_100 = coslat_mean(ep2_100, lat_range=lat_range)
        # Use -ep2 so positive means upward wave activity, matching the old Fz scatter convention.
        out = (-ep2_100).load()
    return out


def ep_window_mean(ep_da: xr.DataArray, date_values, start_end=EP_WINDOW):
    start, end = start_end
    mask = date_mask(date_values, start=start, end=end)
    return ep_da.isel(lead_time=mask).mean("lead_time", skipna=True)


def load_all_wave_metrics(case=CASE, date_values=None, start_end=EP_WINDOW):
    if date_values is None:
        _, date_values = load_hindcast_o3(case)
    rows = []
    series = {}
    for wave in WAVES:
        da = load_epflux_wave(case, wave=wave)
        da = da.isel(lead_time=slice(0, len(date_values)))
        metric = ep_window_mean(da, date_values, start_end=start_end)
        series[wave] = da.assign_coords(date=("lead_time", date_values[: da.sizes["lead_time"]]))
        for member, val in zip(metric["member_short"].values, metric.values):
            rows.append({"member": str(member), "wave": wave, "ep100_upward": float(val)})
    return pd.DataFrame(rows), series


def compute_rmse_table(o3_hind: xr.DataArray, date_h, o3_ref: xr.DataArray, date_ref, start=(1, 1), end=O3_END, label="Jan01-May30"):
    mh = date_mask(date_h, start=start, end=end)
    mr = date_mask(date_ref, start=start, end=end, year=REF_YEAR)
    h = o3_hind.isel(lead_time=mh)
    r = o3_ref.isel(lead_time=mr)
    n = min(h.sizes["lead_time"], r.sizes["lead_time"])
    h = h.isel(lead_time=slice(0, n))
    r = r.isel(lead_time=slice(0, n))
    diff = h - r
    rmse = np.sqrt((diff ** 2).mean("lead_time", skipna=True))
    return pd.DataFrame({
        "member": [str(v) for v in rmse["member_short"].values],
        "RMSE_DU": rmse.values.astype(float),
        "rmse_window": label,
        "n_days": n,
    }).sort_values("RMSE_DU").reset_index(drop=True)


def merge_rmse_ep(rmse_df, ep_metric_df):
    wide = ep_metric_df.pivot(index="member", columns="wave", values="ep100_upward").reset_index()
    wide = wide.rename(columns={w: f"EP100_{w}" for w in WAVES})
    return rmse_df.merge(wide, on="member", how="left")

In [ ]:
# -----------------------------
# Product sanity check for 0008-01
# -----------------------------
expected = {
    "partial_O3": CASE_ROOT / "partial_O3" / f"{CASE}_partial_O3_all_ranges_members.nc",
    "EP all": CASE_ROOT / "EPflux_daily_ubar" / "all_waves" / f"EPFLUX_all_waves_{CASE}_members_time_plev_lat.nc",
    "EP wave1": CASE_ROOT / "EPflux_daily_ubar" / "wave1" / f"EPFLUX_wave1_{CASE}_members_time_plev_lat.nc",
    "EP wave2": CASE_ROOT / "EPflux_daily_ubar" / "wave2" / f"EPFLUX_wave2_{CASE}_members_time_plev_lat.nc",
    "EP rest": CASE_ROOT / "EPflux_daily_ubar" / "wave_rest" / f"EPFLUX_wave_rest_{CASE}_members_time_plev_lat.nc",
    "FWD": CASE_ROOT / "final_warming_date" / f"{CASE}_FWD_plev_member.nc",
    "AO/NAM projection": CASE_ROOT / "NAM_B2000WCN_projection" / f"{CASE}_AO_NAM_B2000WCN_projection_members.nc",
}
for name, path in expected.items():
    print(f"{name:18s}: {path.exists()}  {path}")

with xr.open_dataset(expected["partial_O3"], decode_times=False) as ds:
    print("\npartial_O3 dims:", dict(ds.sizes))
    print("partial_O3 vars:", list(ds.data_vars))
with xr.open_dataset(expected["EP all"], decode_times=False) as ds:
    print("\nEP all dims:", dict(ds.sizes))
    print("EP attrs:", {k: ds.attrs.get(k) for k in ["method", "do_ubar", "use_omega_w_correction"]})

In [ ]:
# -----------------------------
# Figure routing for the split notebook
# -----------------------------
NOTEBOOK_STEM = "Hindcast_0008_01_05_z300_maps_wave_amplitude"

def use_figure_dir(content_tag: str):
    global FIG_DIR
    FIG_DIR = OUT_ROOT / "figures" / NOTEBOOK_STEM / content_tag
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    print("Figure dir:", FIG_DIR)
    return FIG_DIR


In [ ]:
# -----------------------------
# Shared scalar data prep for split diagnostics
# -----------------------------
o3_h, date_h = load_hindcast_o3(CASE)
o3_ref, date_ref = load_bwcn_ref_o3(REF_YEAR)
ep_metric_df, ep_series = load_all_wave_metrics(CASE, date_h, EP_WINDOW)
rmse_all = compute_rmse_table(o3_h, date_h, o3_ref, date_ref, start=(1, 1), end=O3_END, label="Jan01-May30")
rmse_ep_all = merge_rmse_ep(rmse_all, ep_metric_df)
rmse_ep_all["EP100_wave1_plus_wave2"] = rmse_ep_all["EP100_wave1"] + rmse_ep_all["EP100_wave2"]
EP_SCALAR_COLS = [
    ("EP100_all_waves", "all_waves", "All", "black"),
    ("EP100_wave1", "wave1", "W1", "tab:blue"),
    ("EP100_wave2", "wave2", "W2", "tab:orange"),
    ("EP100_wave_rest", "wave_rest", "Rest", "tab:green"),
    ("EP100_wave1_plus_wave2", "wave1_plus_wave2", "W1+W2", "tab:purple"),
]
print("Prepared base tables:", rmse_ep_all.shape)


In [ ]:
# Define Z300 helper functions without running the monthly diagnostic block.
RUN_Z300_DIAGNOSTICS = False


## 图：Z300 月平均波型指标与 EPFlux 分量

**做什么**：把原先单一 Jan20-Feb10 的 Z300 source test 扩展到 Jan、Feb、Mar、Apr、May 月平均，分别计算 ACC、stationary-wave closeness、stationary-wave projection 与同月 EPFlux 分量的 member-to-member 相关；同时单独画 monthly stationary-wave projection vs EP100 wave1+wave2 的散点图。

**怎么做**：每个成员的 Z300 为 300 hPa 高度月平均，区域 20-90N。ACC 是成员 Z300 stationary anomaly 与 BWCN0008 同月 Z300 stationary anomaly 的加权 pattern correlation。stationary-wave closeness 是成员 Z300 stationary anomaly 与 B2000WCN001002 全年份同月气候态 stationary wave target 的加权 pattern correlation。projection 是成员异常投影到同一个 B2000WCN 气候态 stationary wave target 上的加权投影系数，即加权 dot(member anomaly, target) / dot(target, target)。若 `Longrun/date_treatment/Climatology.ipynb` 已生成完整的 `Z3_climatology_plev_doy.nc`，这里会优先从 `Z3_clim_all(doy, plev, lat, lon)` 抽取 300 hPa 并按对应月份/窗口平均，再去掉 zonal mean 作为 B2000WCN stationary-wave target；只有该完整气候态缺失时，才回退到旧缓存或从原始 Z3 现算。

**科学问题**：哪个月份的对流层 300 hPa 波型/投影最能对应进入平流层的 EPFlux 分量？这种关系是 wave1、wave2、rest，还是 all-wave 更明显？MAR 若接近 0，是因为算法退化、目标场异常，还是因为该月 projection 与 EP100 W1+W2 的成员排序确实不一致？

**预期**：如果对流层 planetary wave 是源头，Jan-Feb 或 Apr 的 Z300 climatological-stationary-wave projection 应与 100 hPa EPFlux 的 wave1+wave2 更相关；如果 MAR 不是算法问题，MAR 的 projection 和 EP100 W1+W2 应该都有非零 spread，但二者 scatter 没有线性关系。

**运行后解读**：待图生成后填写。Jan20-Feb10 三个 Z300 指标 vs O3 RMSE 的旧图先保留。


In [ ]:
# -----------------------------
# Z300 tropospheric diagnostics
# -----------------------------

def interp_profile_logp(da_var: xr.DataArray, p_hyb: xr.DataArray, p_tgt_pa: float) -> xr.DataArray:
    target = np.array([float(p_tgt_pa)], dtype=float)
    def _interp_col(vcol, pcol):
        vcol = np.asarray(vcol, dtype=float)
        pcol = np.asarray(pcol, dtype=float)
        mask = np.isfinite(vcol) & np.isfinite(pcol) & (pcol > 0)
        if mask.sum() < 2:
            return np.array([np.nan], dtype=float)
        p = pcol[mask]
        v = vcol[mask]
        idx = np.argsort(p)
        return np.interp(np.log(target), np.log(p[idx]), v[idx], left=np.nan, right=np.nan)
    out = xr.apply_ufunc(
        _interp_col,
        da_var,
        p_hyb,
        input_core_dims=[["lev"], ["lev"]],
        output_core_dims=[["plev"]],
        vectorize=True,
        dask="allowed",
        output_dtypes=[float],
    )
    return out.assign_coords(plev=("plev", target)).isel(plev=0, drop=True)


def window_token(label: str) -> str:
    return re.sub(r"[^A-Za-z0-9]+", "_", label).strip("_")


def z300_from_file(path: Path, start_end=Z300_WINDOW, lat_range=LAT_Z300) -> xr.DataArray:
    with xr.open_dataset(path, decode_times=False) as ds:
        date = np.asarray(ds["date"].values, dtype=np.int64)
        mask = date_mask(date, start=start_end[0], end=start_end[1])
        ds = ds.isel(time=mask)
        ds = ds.sel(lat=slice(lat_range[0], lat_range[1]))
        p_mid = ds["hyam"] * ds["P0"] + ds["hybm"] * ds["PS"]
        z = interp_profile_logp(ds["Z3"].transpose("time", "lat", "lon", "lev"), p_mid.transpose("time", "lat", "lon", "lev"), PLEV_Z300_PA)
        z_mean = z.mean("time", skipna=True).load()
    z_mean.name = "Z300"
    z_mean.attrs.update({"units": "m", "plev_pa": PLEV_Z300_PA, "window": str(start_end)})
    return z_mean


def build_z300_hindcast_cache(case=CASE, start_end=Z300_WINDOW, label="Cwindow", overwrite=False):
    out = CACHE_DIR / f"Z300_{case}_members_{window_token(label)}.nc"
    if out.exists() and not overwrite:
        return xr.open_dataarray(out).load()
    files = sorted((HINDCAST_ROOT / case / "Z3").glob("*.Z3.nc"))
    if not files:
        raise FileNotFoundError(f"No Z3 files for {case}")
    das, mids = [], []
    for f in files:
        mid = member_short_id(f.name)
        print("Z300", case, label, mid)
        das.append(z300_from_file(f, start_end=start_end))
        mids.append(mid)
    da = xr.concat(das, dim=pd.Index(mids, name="member"))
    da.to_netcdf(out)
    print(f"Saved cache: {out}")
    return da


def build_z300_bwcn_ref_cache(year=REF_YEAR, start_end=Z300_WINDOW, label="Cwindow", overwrite=False):
    out = CACHE_DIR / f"Z300_BWCN_{year:04d}_{window_token(label)}.nc"
    if out.exists() and not overwrite:
        return xr.open_dataarray(out).load()
    f = BWCN_ROOT / "Z3" / f"BWCN.cam.h3.{year:04d}.Z3.nc"
    da = z300_from_file(f, start_end=start_end)
    da.to_netcdf(out)
    print(f"Saved cache: {out}")
    return da


def doy_values_for_window(start_end):
    month_lengths = np.array([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31], dtype=int)
    (sm, sd), (em, ed) = start_end
    start_doy = int(month_lengths[: sm - 1].sum() + sd)
    end_doy = int(month_lengths[: em - 1].sum() + ed)
    if end_doy < start_doy:
        raise ValueError(f"Window crosses year boundary, unsupported here: {start_end}")
    return np.arange(start_doy, end_doy + 1, dtype=np.int16)


def z300_target_from_full_z3_climatology(clim_file: Path, label, start_end) -> xr.DataArray:
    doys = doy_values_for_window(start_end)
    stat = clim_file.stat()
    with xr.open_dataset(clim_file, decode_times=False) as ds:
        if "Z3_clim_all" not in ds.data_vars:
            raise KeyError(f"{clim_file} does not contain Z3_clim_all")
        z = ds["Z3_clim_all"].sel(plev=PLEV_Z300_PA, method="nearest")
        available = np.intersect1d(np.asarray(z["doy"].values, dtype=np.int16), doys)
        if len(available) == 0:
            raise ValueError(f"No requested DOYs {doys[0]}-{doys[-1]} found in {clim_file}")
        z = z.sel(doy=available).mean("doy", skipna=True)
        z = select_latband(z, LAT_Z300).load()
    target = (z - z.mean("lon", skipna=True)).astype(np.float32)
    target.name = f"Z300_B2000_stationary_wave_target_{window_token(label)}"
    target.attrs.update({
        "definition": "B2000WCN001002 all-year Z3_clim_all at 300 hPa averaged over the requested window; zonal mean removed",
        "source_file": str(clim_file),
        "source_file_size": int(stat.st_size),
        "source_file_mtime_ns": int(stat.st_mtime_ns),
        "source_variable": "Z3_clim_all",
        "window_label": str(label),
        "start_end": str(start_end),
        "doy_start": int(doys[0]),
        "doy_end": int(doys[-1]),
        "plev_pa": float(PLEV_Z300_PA),
        "lat_range": str(LAT_Z300),
    })
    return target


def cache_matches_full_z3_climatology(cache_file: Path, source_file: Path, start_end) -> bool:
    if not cache_file.exists() or cache_file.stat().st_size <= 0:
        return False
    try:
        stat = source_file.stat()
        with xr.open_dataarray(cache_file, decode_times=False) as da:
            return (
                da.attrs.get("source_file") == str(source_file)
                and da.attrs.get("start_end") == str(start_end)
                and int(da.attrs.get("source_file_size", -1)) == int(stat.st_size)
                and int(da.attrs.get("source_file_mtime_ns", -1)) == int(stat.st_mtime_ns)
            )
    except Exception:
        return False


def build_z300_b2000_stationary_target(label, start_end, overwrite=False, max_years=CLIM_MAX_B2000_YEARS_FOR_Z300):
    out = CACHE_DIR / f"Z300_B2000WCN001002_{window_token(label)}_stationary_wave_target.nc"
    full_z3_clim_file = B2000_ROOT / "climatology" / "Z3_climatology_plev_doy.nc"

    if full_z3_clim_file.exists():
        if not overwrite and cache_matches_full_z3_climatology(out, full_z3_clim_file, start_end):
            return xr.open_dataarray(out).load()
        target = z300_target_from_full_z3_climatology(full_z3_clim_file, label, start_end)
        target.to_netcdf(out)
        print(f"Saved cache from full Z3 climatology: {out}")
        return target

    if out.exists() and not overwrite:
        return xr.open_dataarray(out).load()

    monthly_clim_file = B2000_ROOT / "climatology" / "Z300_monthly_stationary_wave_climatology.nc"
    if label in MONTH_ORDER and monthly_clim_file.exists():
        month_num = MONTH_ORDER.index(label) + 1
        with xr.open_dataset(monthly_clim_file, decode_times=False) as ds:
            target = ds["Z300_stationary_wave"].sel(month=month_num).load()
        target.name = f"Z300_B2000_stationary_wave_target_{window_token(label)}"
        target.attrs.update({
            "definition": "Legacy B2000WCN001002 monthly 300 hPa stationary-wave target; zonal mean removed",
            "source_file": str(monthly_clim_file),
            "month": label,
        })
        target.to_netcdf(out)
        print(f"Saved cache from legacy monthly Z300 climatology: {out}")
        return target

    files = sorted((B2000_ROOT / "Z3").glob("B2000WCN.sample.cam.h3.*.Z3.nc"))
    if max_years is not None:
        files = files[: int(max_years)]
    if not files:
        raise FileNotFoundError(f"No B2000WCN001002 Z3 files under {B2000_ROOT / 'Z3'}")
    das = []
    for f in files:
        print("Z300 B2000 climatology", label, f.name)
        das.append(z300_from_file(f, start_end=start_end, lat_range=LAT_Z300))
    clim = xr.concat(das, dim="year_index").mean("year_index", skipna=True)
    target = clim - clim.mean("lon", skipna=True)
    target.name = f"Z300_B2000_stationary_wave_target_{window_token(label)}"
    target.attrs.update({
        "definition": "Fallback B2000WCN001002 all-year climatological stationary wave at 300 hPa; zonal mean removed",
        "window": str(start_end),
        "lat_range": str(LAT_Z300),
        "source_root": str(B2000_ROOT),
        "max_years": "all" if max_years is None else int(max_years),
    })
    target.to_netcdf(out)
    print(f"Saved cache from raw Z3 fallback: {out}")
    return target

def weighted_pattern_corr(a: xr.DataArray, b: xr.DataArray, lat_range=LAT_Z300, remove_zonal_mean=True):
    a, b = xr.align(select_latband(a, lat_range), select_latband(b, lat_range), join="inner")
    if remove_zonal_mean:
        a = a - a.mean("lon", skipna=True)
        b = b - b.mean("lon", skipna=True)
    w = np.sqrt(np.cos(np.deg2rad(a["lat"])).clip(0, 1))
    aw = (a * w).values.ravel()
    bw = (b * w).values.ravel()
    mask = np.isfinite(aw) & np.isfinite(bw)
    if mask.sum() < 10:
        return np.nan
    return float(np.corrcoef(aw[mask], bw[mask])[0, 1])


def weighted_projection(a: xr.DataArray, target: xr.DataArray, lat_range=LAT_Z300):
    a, target = xr.align(select_latband(a, lat_range), select_latband(target, lat_range), join="inner")
    a = a - a.mean("lon", skipna=True)
    target = target - target.mean("lon", skipna=True)
    w = np.cos(np.deg2rad(a["lat"])).clip(0, 1)
    num = (a * target * w).sum(skipna=True)
    den = (target * target * w).sum(skipna=True)
    return float((num / den).values)


def ep_wide_for_window(start_end):
    df, _ = load_all_wave_metrics(CASE, date_h, start_end=start_end)
    wide = df.pivot(index="member", columns="wave", values="ep100_upward").reset_index()
    wide = wide.rename(columns={w: f"EP100_{w}" for w in WAVES})
    wide["EP100_wave1_plus_wave2"] = wide["EP100_wave1"] + wide["EP100_wave2"]
    return wide


def z300_metric_table_for_window(label, start_end):
    members = build_z300_hindcast_cache(CASE, start_end=start_end, label=label)
    ref = build_z300_bwcn_ref_cache(REF_YEAR, start_end=start_end, label=label)
    target = build_z300_b2000_stationary_target(label, start_end)
    rows = []
    for mid in members["member"].values:
        z = members.sel(member=mid)
        rows.append({
            "member": str(mid),
            "window": label,
            "Z300_ACC_to_BWCN0008": weighted_pattern_corr(z, ref),
            "Z300_stationary_closeness": weighted_pattern_corr(z, target),
            "Z300_stationary_projection": weighted_projection(z, target),
        })
    return pd.DataFrame(rows)


def z300_ep_correlation_rows(metric_df, ep_df, window_label):
    joined = metric_df.merge(ep_df, on="member", how="inner")
    rows = []
    for z_col in ["Z300_ACC_to_BWCN0008", "Z300_stationary_closeness", "Z300_stationary_projection"]:
        for ep_col, key, ep_label, _ in EP_SCALAR_COLS:
            r, p = finite_corr(joined[z_col], joined[ep_col])
            rows.append({"window": window_label, "z300_metric": z_col, "ep_metric": key, "R": r, "P": p})
    return rows, joined


def plot_monthly_z300_heatmap(corr_df, z_metric, title, filename):
    sub = corr_df[corr_df["z300_metric"] == z_metric].copy()
    mat = sub.pivot(index="window", columns="ep_metric", values="R").reindex(index=MONTH_ORDER)
    mat = mat[["all_waves", "wave1", "wave2", "wave_rest", "wave1_plus_wave2"]]
    fig, ax = plt.subplots(figsize=(8.8, 4.8), constrained_layout=True)
    im = ax.imshow(mat.values, cmap="RdBu_r", vmin=-0.8, vmax=0.8, aspect="auto")
    ax.set_xticks(np.arange(mat.shape[1]))
    ax.set_xticklabels(["All", "W1", "W2", "Rest", "W1+W2"])
    ax.set_yticks(np.arange(mat.shape[0]))
    ax.set_yticklabels(mat.index.tolist())
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            val = mat.values[i, j]
            ax.text(j, i, f"{val:.2f}" if np.isfinite(val) else "NA", ha="center", va="center", fontsize=9)
    ax.set_title(title)
    ax.set_xlabel("EP100 component in same month")
    ax.set_ylabel("Z300 monthly mean")
    fig.colorbar(im, ax=ax, label="Across-member correlation R")
    savefig(fig, filename)
    plt.show()


def z300_monthly_projection_sanity(monthly_values: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for month in MONTH_ORDER:
        sub = monthly_values[monthly_values["window"] == month].copy()
        x = sub["Z300_stationary_projection"].astype(float)
        y = sub["EP100_wave1_plus_wave2"].astype(float)
        r, p = finite_corr(x, y)
        target = build_z300_b2000_stationary_target(month, MONTH_WINDOWS[month])
        target_eddy = target - target.mean("lon", skipna=True)
        rows.append({
            "window": month,
            "n_members": int(np.isfinite(x).sum()),
            "projection_mean": float(np.nanmean(x)),
            "projection_std": float(np.nanstd(x, ddof=1)),
            "projection_min": float(np.nanmin(x)),
            "projection_max": float(np.nanmax(x)),
            "ep100_wave1_plus_wave2_mean": float(np.nanmean(y)),
            "ep100_wave1_plus_wave2_std": float(np.nanstd(y, ddof=1)),
            "target_stationary_std_m": float(np.nanstd(target_eddy.values)),
            "R_projection_vs_EP100_wave1_plus_wave2": r,
            "P_projection_vs_EP100_wave1_plus_wave2": p,
            "sanity_note": "Near-zero R is acceptable if projection_std, EP_std, and target_std are non-zero.",
        })
    return pd.DataFrame(rows)


def plot_stationary_projection_wave12_scatter(monthly_values: pd.DataFrame, months=("Jan", "Feb", "Mar", "Apr")):
    ncols = 2
    nrows = int(np.ceil(len(months) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(10.8, 8.2), constrained_layout=True)
    axes = np.asarray(axes).ravel()
    for ax, month in zip(axes, months):
        sub = monthly_values[monthly_values["window"] == month].copy()
        scatter_fit(
            ax,
            sub,
            "Z300_stationary_projection",
            "EP100_wave1_plus_wave2",
            title=month,
            xlabel="Z300 projection",
            ylabel="EP100 W1+W2",
            color="tab:purple",
            annotate_members=False,
        )
    for ax in axes[len(months):]:
        ax.set_visible(False)
    fig.suptitle("Monthly stationary-wave projection vs same-month EP100 W1+W2", fontsize=11)
    savefig(fig, f"{CASE}_Z300_monthly_stationary_projection_vs_EP100_wave1plus2_scatter")
    plt.show()


if RUN_Z300_DIAGNOSTICS:
    monthly_corr_rows = []
    monthly_joined = []
    for month in MONTH_ORDER:
        metric_df = z300_metric_table_for_window(month, MONTH_WINDOWS[month])
        ep_df = ep_wide_for_window(MONTH_WINDOWS[month])
        rows, joined = z300_ep_correlation_rows(metric_df, ep_df, month)
        monthly_corr_rows.extend(rows)
        monthly_joined.append(joined.assign(window=month))
    z300_monthly_corr = pd.DataFrame(monthly_corr_rows)
    z300_monthly_values = pd.concat(monthly_joined, ignore_index=True)
    z300_monthly_corr.to_csv(TABLE_DIR / f"{CASE}_Z300_monthly_metric_EPFlux_correlations.csv", index=False)
    z300_monthly_values.to_csv(TABLE_DIR / f"{CASE}_Z300_monthly_metric_EPFlux_values.csv", index=False)
    print(z300_monthly_corr.head())

    z300_projection_sanity = z300_monthly_projection_sanity(z300_monthly_values)
    z300_projection_sanity.to_csv(TABLE_DIR / f"{CASE}_Z300_monthly_stationary_projection_sanity.csv", index=False)
    print("\nStationary projection sanity check:")
    print(z300_projection_sanity[[
        "window",
        "projection_std",
        "ep100_wave1_plus_wave2_std",
        "target_stationary_std_m",
        "R_projection_vs_EP100_wave1_plus_wave2",
    ]])

    plot_monthly_z300_heatmap(
        z300_monthly_corr,
        "Z300_ACC_to_BWCN0008",
        "Monthly Z300 ACC vs EP100 components",
        f"{CASE}_Z300_ACC_vs_EPFlux_components",
    )
    plot_monthly_z300_heatmap(
        z300_monthly_corr,
        "Z300_stationary_closeness",
        "Monthly stationary-wave closeness vs EP100",
        f"{CASE}_Z300_stationary_closeness_vs_EPFlux_components",
    )
    plot_monthly_z300_heatmap(
        z300_monthly_corr,
        "Z300_stationary_projection",
        "Monthly stationary-wave projection vs EP100",
        f"{CASE}_Z300_stationary_projection_vs_EPFlux_components",
    )
    plot_stationary_projection_wave12_scatter(z300_monthly_values, months=("Jan", "Feb", "Mar", "Apr"))

    # Keep the original Jan20-Feb10 source metrics vs O3 RMSE diagnostic for now.
    c_metric = z300_metric_table_for_window("Jan20_Feb10", Z300_WINDOW)
    z300_join = rmse_ep_all.merge(c_metric.drop(columns=["window"]), on="member", how="left")
    z300_join["EP100_wave1_plus_wave2"] = z300_join["EP100_wave1"] + z300_join["EP100_wave2"]
    z300_join.to_csv(TABLE_DIR / f"{CASE}_Z300_tropospheric_metrics.csv", index=False)
    print(z300_join.head())

    fig, axes = plt.subplots(1, 3, figsize=(13.8, 4.8), constrained_layout=True)
    rmse_specs = [
        ("Z300_ACC_to_BWCN0008", "ACC", "tab:olive"),
        ("Z300_stationary_closeness", "Closeness", "tab:cyan"),
        ("Z300_stationary_projection", "Projection", "tab:brown"),
    ]
    for ax, (xcol, label, color) in zip(axes, rmse_specs):
        scatter_fit(ax, z300_join, xcol, "RMSE_DU", title=f"{label} vs RMSE", xlabel=label, ylabel="O3 RMSE (DU)", color=color, annotate_members=False)
    fig.suptitle("Jan20-Feb10 Z300 source metrics vs O3 RMSE", fontsize=11)
    savefig(fig, f"{CASE}_Z300_tropospheric_signal_tests")
    plt.show()
else:
    print("RUN_Z300_DIAGNOSTICS is False; skipping Z300 cache build and scatter plots.")


## 图：Z300 monthly pointwise correlation maps

**做什么**：分别为 Jan、Feb、Mar、Apr、May 画五张 Z300 pointwise correlation map。每张图包含六个子图：同月 EP100 all waves、wave1、wave2、wave1+wave2、wave rest，以及整季 O3 RMSE。

**怎么做**：每个格点上，用 30 个成员的 Z300 monthly anomaly（先减 ensemble mean）与一个成员标量做相关。EP100 指标是同月 `mean(-ep2)`，即 100 hPa、40-80N 平均的垂直 EP-flux 分量，不是 divergence。RMSE 仍是 `60-90N, 30-70 hPa` O3 的 Jan1-May30 RMSE。底图使用与 `WindEpfluxEddyheatflux_Structure.ipynb` 一致的 `NorthPolarStereo`，填色为相关系数，黑色等值线为 B2000WCN001002 同月 300 hPa climatological stationary waves（Z300 气候态去 zonal mean）。

**科学问题**：哪些对流层 300 hPa 区域的成员间高度异常，与同月进入平流层的 wave1/wave2/wave-rest EPFlux 或最终 O3 RMSE 同变？这些局地相关是否贴近 climatological stationary-wave ridge/trough？

**预期**：如果对流层 stationary-wave geometry 是源头，相关高值应在气候态 stationary-wave 大振幅区附近成片出现，并且 W1+W2 图应比单独 wave1/wave2 更接近 all waves。

**运行后解读**：待图生成后填写。


In [ ]:
use_figure_dir("01_monthly_pointwise_corr_maps")


In [ ]:
# -----------------------------
# Source-hunting idea 1: monthly Z300 pointwise correlation maps across members
# Colors are local member-to-member correlations. Black contours are the B2000WCN001002
# climatological stationary waves at 300 hPa for the same month/window.
# -----------------------------

def pointwise_member_corr(field_member_lat_lon: xr.DataArray, metric_by_member: pd.Series) -> xr.DataArray:
    da = field_member_lat_lon.copy()
    member_short = [member_short_id(v) for v in da["member"].values]
    da = da.assign_coords(member_short=("member", member_short)).swap_dims({"member": "member_short"})
    common = [m for m in da["member_short"].values if m in metric_by_member.index]
    da = da.sel(member_short=common)
    x = np.asarray([metric_by_member.loc[m] for m in common], dtype=float)
    x = xr.DataArray(x, dims="member_short", coords={"member_short": common})
    x_anom = x - x.mean("member_short")
    y_anom = da - da.mean("member_short", skipna=True)
    cov = (x_anom * y_anom).mean("member_short", skipna=True)
    corr = cov / (x_anom.std("member_short") * y_anom.std("member_short", skipna=True))
    corr.name = "member_correlation"
    return corr


try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    from cartopy.util import add_cyclic_point

    HAS_CARTOPY = True
except ImportError:
    HAS_CARTOPY = False
    ccrs = None
    cfeature = None
    add_cyclic_point = None


def map_lon_lat_values(da: xr.DataArray):
    """Return 2-D lat-lon data in a Cartopy-safe -180..180 longitude convention."""
    lon = np.asarray(da["lon"].values, dtype=float)
    lat = np.asarray(da["lat"].values, dtype=float)
    values = np.asarray(da.transpose("lat", "lon").values, dtype=float)
    if lat[0] > lat[-1]:
        lat = lat[::-1]
        values = values[::-1, :]

    lon_wrapped = ((lon + 180.0) % 360.0) - 180.0
    order = np.argsort(lon_wrapped)
    lon_sorted = lon_wrapped[order]
    values_sorted = values[:, order]

    if add_cyclic_point is None:
        return lon_sorted, lat, values_sorted
    try:
        values_cyc, lon_cyc = add_cyclic_point(values_sorted, coord=lon_sorted)
        return lon_cyc, lat, values_cyc
    except ValueError:
        return lon_sorted, lat, values_sorted


def add_polar_map_features(ax, data_crs):
    try:
        ax.add_feature(cfeature.COASTLINE.with_scale("110m"), linewidth=0.55, edgecolor="0.25")
        ax.add_feature(cfeature.BORDERS.with_scale("110m"), linewidth=0.28, edgecolor="0.35", alpha=0.75)
        ax.gridlines(crs=data_crs, linewidth=0.32, color="0.45", alpha=0.45, linestyle="--", draw_labels=False)
    except Exception as exc:
        ax.text(
            0.02,
            0.02,
            f"map features unavailable: {type(exc).__name__}",
            transform=ax.transAxes,
            fontsize=7,
            ha="left",
            va="bottom",
            color="0.35",
        )


def plot_corr_map_with_stationary_contours(
    corr_da: xr.DataArray,
    stationary_da: xr.DataArray,
    ax,
    title,
    vlim=0.8,
    wave_levels=np.arange(-240, 241, 40),
):
    lon, lat, corr_values = map_lon_lat_values(corr_da)
    _, _, wave_values = map_lon_lat_values(stationary_da)
    levels = np.linspace(-vlim, vlim, 17)
    if HAS_CARTOPY:
        data_crs = ccrs.PlateCarree()
        cf = ax.contourf(lon, lat, corr_values, levels=levels, cmap="RdBu_r", extend="both", transform=data_crs)
        ax.contour(lon, lat, wave_values, levels=wave_levels, colors="k", linewidths=0.62, alpha=0.84, transform=data_crs)
        ax.set_extent([-180.0, 180.0, LAT_Z300[0], LAT_Z300[1]], crs=data_crs)
        add_polar_map_features(ax, data_crs)
    else:
        cf = ax.contourf(lon, lat, corr_values, levels=levels, cmap="RdBu_r", extend="both")
        ax.contour(lon, lat, wave_values, levels=wave_levels, colors="k", linewidths=0.62, alpha=0.84)
        ax.set_xlim(-180, 180)
        ax.set_ylim(*LAT_Z300)
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")
    ax.set_title(title, fontsize=8)
    return cf


def monthly_pointwise_metrics(month, start_end):
    ep_month = ep_wide_for_window(start_end)
    metrics = ep_month.merge(rmse_ep_all[["member", "RMSE_DU"]], on="member", how="left")
    metrics["EP100_wave1_plus_wave2"] = metrics["EP100_wave1"] + metrics["EP100_wave2"]
    z300_month = build_z300_hindcast_cache(CASE, start_end=start_end, label=month)
    z300_anom = z300_month - z300_month.mean("member", skipna=True)
    stationary = build_z300_b2000_stationary_target(month, start_end)
    corr_maps = {
        col: pointwise_member_corr(z300_anom, metrics.set_index("member")[col])
        for col in ["EP100_all_waves", "EP100_wave1", "EP100_wave2", "EP100_wave1_plus_wave2", "EP100_wave_rest", "RMSE_DU"]
    }
    ds_maps = xr.Dataset({col: da for col, da in corr_maps.items()})
    out = CACHE_DIR / f"Z300_pointwise_corr_maps_{CASE}_{window_token(month)}.nc"
    ds_maps.to_netcdf(out)
    print("Saved:", out)
    return corr_maps, stationary


def plot_monthly_pointwise_corr_maps(month, start_end):
    corr_maps, stationary = monthly_pointwise_metrics(month, start_end)
    metric_cols = ["EP100_all_waves", "EP100_wave1", "EP100_wave2", "EP100_wave1_plus_wave2", "EP100_wave_rest", "RMSE_DU"]
    titles = ["All waves", "Wave 1", "Wave 2", "Wave 1 + Wave 2", "Wave rest", "O3 RMSE"]
    subplot_kw = {"projection": ccrs.NorthPolarStereo()} if HAS_CARTOPY else {}
    fig, axes = plt.subplots(2, 3, figsize=(11.8, 8.4), constrained_layout=True, subplot_kw=subplot_kw)
    cf = None
    for ax, col, title in zip(axes.ravel(), metric_cols, titles):
        cf = plot_corr_map_with_stationary_contours(corr_maps[col], stationary, ax, title)
    cbar = fig.colorbar(cf, ax=axes.ravel().tolist(), shrink=0.76, pad=0.02)
    cbar.set_label("Correlation r", fontsize=8)
    cbar.ax.tick_params(labelsize=8)
    fig.suptitle(
        f"{CASE} {month}: Z300 pointwise r; contours = B2000 stationary waves",
        fontsize=10,
    )
    savefig(fig, f"{CASE}_Z300_pointwise_corr_maps_{window_token(month)}_EPFlux_RMSE")
    plt.show()


for month in MONTH_ORDER:
    plot_monthly_pointwise_corr_maps(month, MONTH_WINDOWS[month])


## 图：Z300 zonal wavenumber amplitude

**做什么**：把 Jan20-Feb10 的 Z300 stationary anomaly 按经向 Fourier 分解，计算 wave1、wave2、wave3-6 振幅，并与 EPFlux/O3 RMSE 比较。

**怎么做**：先对每个成员的 300 hPa 高度场去掉 zonal mean，再沿经度取 Fourier wave-k 振幅，最后在 20-90N 做 cos-lat 加权平均。`synoptic_3_6` 是 wave3-6 振幅平方和开根号。

**科学问题**：这是一个“对流层波源/波型强度”诊断：成员间对流层 300 hPa 的长波振幅，是否已经预示了后续 100 hPa EPFlux 的 wave1/wave2 差异？它回答的是源头波型是否存在，而不是波活动是否真的向上传播。

**和直接比较 300 hPa EPFlux Fz vs 100 hPa EPFlux Fz 的区别**：300-to-100 hPa EPFlux 对比更直接检验垂直传播/耦合链条；Z300 wave amplitude 则更上游，检验对流层 stationary-wave 几何结构和振幅本身。前者更接近“波活动是否传上去”，后者更接近“对流层有没有提供 wave1/wave2 源”。两者应该并行使用：若 Z300 wave2 amp 与 100 hPa wave2 EPFlux 相关，而 300 hPa EPFlux 与 100 hPa EPFlux 也相关，证据链会更完整。

**论文依据**：行星波从对流层进入冬季平流层的基本传播理论可参考 Charney and Drazin (1961), DOI `10.1029/JZ066i001p00083`；stationary planetary wave 垂直传播与极涡扰动可参考 Matsuno (1970), DOI `10.1175/1520-0469(1970)027<0871:VPOSPW>2.0.CO;2`；stationary wave activity flux 诊断可参考 Plumb (1985), DOI `10.1175/1520-0469(1985)042<0217:OTTDPO>2.0.CO;2`；上行 wave activity flux 作为极端平流层事件前兆可参考 Polvani and Waugh (2004), DOI `10.1175/1520-0442(2004)017<3548:UWAFAA>2.0.CO;2`。

**预期**：已有初步结果显示 Z300 wave2 amplitude 与 EP100 wave2 相关最强；这支持 wave2 相关源头假设，但还不直接证明因果。

**运行后解读**：待图生成后填写。


In [ ]:
use_figure_dir("02_zonal_wave_amplitudes")


In [ ]:
# Shared C-window Z300 member field used by the wave-amplitude block.
z300_members = build_z300_hindcast_cache(CASE, start_end=Z300_WINDOW, label="Jan20_Feb10")
print("Prepared z300_members:", z300_members.sizes)


In [ ]:
# -----------------------------
# Source-hunting idea 2: Z300 zonal-wavenumber amplitudes
# Wave-k amplitude is computed from the C-window 300 hPa height field after removing the zonal mean,
# using longitude Fourier coefficients at each latitude and cos-lat averaging over 20-90N.
# Synoptic 3-6 amplitude is sqrt(wave3^2 + wave4^2 + wave5^2 + wave6^2).
# EP100 is the same scalar -ep2 metric: 100 hPa, 40-80N, Jan20-Feb10; it is not divergence.
# -----------------------------

def z300_wave_amplitude(da: xr.DataArray, k: int, lat_range=LAT_Z300) -> float:
    sub = select_latband(da, lat_range)
    arr = sub.values
    lon = np.deg2rad(sub["lon"].values)
    cos_term = np.cos(k * lon)
    sin_term = np.sin(k * lon)
    a = np.nanmean(arr * cos_term[None, :], axis=1) * 2.0
    b = np.nanmean(arr * sin_term[None, :], axis=1) * 2.0
    amp_lat = np.sqrt(a * a + b * b)
    w = np.cos(np.deg2rad(sub["lat"].values)).clip(0, 1)
    return float(np.nansum(amp_lat * w) / np.nansum(w))

z_rows = []
for mid in z300_members["member"].values:
    z = z300_members.sel(member=mid)
    z_anom_lon = z - z.mean("lon", skipna=True)
    row = {"member": member_short_id(mid)}
    for k in [1, 2, 3, 4, 5, 6]:
        row[f"Z300_wave{k}_amp"] = z300_wave_amplitude(z_anom_lon, k)
    row["Z300_synoptic_3_6_amp"] = np.sqrt(sum(row[f"Z300_wave{k}_amp"] ** 2 for k in [3, 4, 5, 6]))
    z_rows.append(row)
z_wave_df = pd.DataFrame(z_rows)
z_wave_join = rmse_ep_all.merge(z_wave_df, on="member", how="left")
z_wave_join["EP100_wave1_plus_wave2"] = z_wave_join["EP100_wave1"] + z_wave_join["EP100_wave2"]
z_wave_join.to_csv(TABLE_DIR / f"{CASE}_Z300_wave_amplitude_metrics.csv", index=False)

amp_corr_rows = []
for amp_col in ["Z300_wave1_amp", "Z300_wave2_amp", "Z300_synoptic_3_6_amp"]:
    for target_col in ["EP100_all_waves", "EP100_wave1", "EP100_wave2", "EP100_wave1_plus_wave2", "EP100_wave_rest", "RMSE_DU"]:
        r, p = finite_corr(z_wave_join[amp_col], z_wave_join[target_col])
        amp_corr_rows.append({"z300_metric": amp_col, "target": target_col, "R": r, "P": p})
amp_corr = pd.DataFrame(amp_corr_rows)
amp_corr.to_csv(TABLE_DIR / f"{CASE}_Z300_wave_amplitude_correlations.csv", index=False)
print(amp_corr.sort_values("P").head(12))

fig, axes = plt.subplots(2, 3, figsize=(16, 9), constrained_layout=True)
plot_pairs = [
    ("Z300_wave1_amp", "EP100_wave1"),
    ("Z300_wave2_amp", "EP100_wave2"),
    ("Z300_wave1_amp", "EP100_wave1_plus_wave2"),
    ("Z300_synoptic_3_6_amp", "EP100_wave_rest"),
    ("Z300_wave1_amp", "RMSE_DU"),
    ("Z300_synoptic_3_6_amp", "RMSE_DU"),
]
for ax, (xcol, ycol) in zip(axes.ravel(), plot_pairs):
    scatter_fit(
        ax,
        z_wave_join,
        xcol,
        ycol,
        f"{xcol} vs {ycol}",
        xlabel=xcol,
        ylabel=ycol,
        color="tab:brown",
        annotate_members=False,
    )
fig.suptitle(
    "Z300 zonal-wavenumber amplitudes: Fourier amplitude of C-window 300 hPa height stationary anomalies",
    fontsize=11,
)
savefig(fig, f"{CASE}_Z300_wave_amplitudes_vs_EPFlux_RMSE")
plt.show()